In [1]:
import numpy as np

import matplotlib.pyplot as plt
import ipywidgets as widgets

import albumentations as albu

from utils.misc import parse_args
from dataset import *
from dataset.config import *

In [2]:
def display_image(array: np.ndarray):
    """
    3D NumPy 배열을 슬라이서 위젯을 사용하여 표시
    """
    # 슬라이드를 사용하여 슬라이스를 스크롤
    def view_image(slice_index):
        plt.figure(figsize=(10, 10))  # 이미지를 표시할 Figure를 설정
        plt.imshow(array[slice_index], cmap='gray')  # 현재 슬라이스 이미지를 회색조로 표시
        plt.title(f'Slice {slice_index}')  # 슬라이스 번호를 제목으로 설정
        plt.show()  # 이미지를 출력

    slice_slider = widgets.IntSlider(min=0, max=array.shape[0] - 1, step=1, description='Slice:')  # 슬라이더를 생성
    widgets.interact(view_image, slice_index=slice_slider)  # 슬라이더와 view_image 함수를 연결하여 상호작용

In [8]:
strides = [stride for stride in range(213, 213, 32)]
strides

[]

In [3]:
args = parse_args([
    "--data_path", "F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/",
    "--model", "resnet18",
    "-bs", "16",
    "--max_epoch", "10",
    "--fp16",
    ])
print("Setting Arguments.. : ", args)

Setting Arguments.. :  Namespace(seed=42, img_size=416, img_crop_size=288, eval_first=False, tfboard=False, save_folder='./checkpoints/', vis_tgt=False, vis_aux_loss=False, fp16=True, batch_size=16, max_epoch=10, wp_epoch=1, eval_epoch=10, no_aug_epoch=20, model='resnet18', conf_thresh=0.005, nms_thresh=0.6, topk=1000, pretrained=None, resume=None, nms_class_agnostic=False, input_channels=32, model_weight_path=None, data_path='F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/', dataset='voc', load_cache=False, num_workers=4, label_file_name='train.csv', label_weight_priority=[], preprocess_data_folder_name='raw_data/', multi_scale=False, ema=False, min_box_size=8.0, mosaic=None, mixup=None, grad_accumulate=1, distributed=False, dist_url='env://', world_size=1, sybn=False, debug=False)


In [4]:
data_config = SeriesDataConfig(args)
build_raw_data(args)

In [5]:
train_transform = build_transform(args)
train_dataset = build_dataset(args, data_config, train_transform, is_train=True)

In [ ]:
NOISE_PROB = 1.0
# NOISE_PROB = 0.125
albu_noise_set = [
    albu.GaussNoise(p=NOISE_PROB),
    albu.ShotNoise(p=NOISE_PROB),
    albu.AdditiveNoise(p=NOISE_PROB),
    albu.MultiplicativeNoise(p=NOISE_PROB)
    ]

COLOR_MOD_PROB = 0.5
HUE_RATE = 0.015625
HUE_DEGREE = 360 * HUE_RATE
albu_color_mod_set = [
    albu.ColorJitter(p=COLOR_MOD_PROB, hue=(-HUE_RATE, HUE_RATE)),
    albu.HueSaturationValue(p=COLOR_MOD_PROB, hue_shift_limit=(-HUE_DEGREE, HUE_DEGREE)),
    ]

BLUR_PROB = 1.0
# BLUR_PROB = 0.125
albu_blur_set = [
    albu.Blur(p=BLUR_PROB),
    albu.GaussianBlur(p=BLUR_PROB),
    albu.MedianBlur(p=BLUR_PROB),
    albu.MotionBlur(p=BLUR_PROB),
    albu.ZoomBlur(p=BLUR_PROB),
    albu.GlassBlur(p=BLUR_PROB),
    albu.AdvancedBlur(p=BLUR_PROB)
    ]

EQUALIZE_PROB = 1.0
# EQUALIZE_PROB = 0.25
albu_equalize_set = [
    albu.Equalize(p=EQUALIZE_PROB),
    albu.CLAHE(p=EQUALIZE_PROB),
    ]

In [ ]:
albu.Rotate((-15, 15), p=1.0)

In [ ]:
def get_rand_albu_transform() -> albu.Compose:
    np.random.shuffle(albu_noise_set)
    # np.random.shuffle(albu_color_mod_set)
    np.random.shuffle(albu_blur_set)
    np.random.shuffle(albu_equalize_set)
    
    return albu.Compose(
        [albu_noise_set[0]
        #  , albu_color_mod_set[0]
         , albu_blur_set[0]
         , albu_equalize_set[0]
         , albu.Rotate((-15, 15), p=1.0)
         ])

albu_transform = get_rand_albu_transform()

In [12]:
image, label = train_dataset.__getitem__(1234)
image = image.numpy()
image.shape

(32, 288, 288)

In [ ]:
display_image(image)

interactive(children=(IntSlider(value=0, description='Slice:', max=31), Output()), _dom_classes=('widget-inter…